## Data Curation Notebook (v2) — Centralized DDI Dataset Construction

### Purpose
This notebook is the **single source of truth** for building the curated drug-drug interaction (DDI)
datasets used downstream in this project. It consolidates logic that was previously spread across
several exploratory notebooks in `notebooks_test/` into one documented, linear, reproducible pipeline.

### What this notebook does, end to end
1. **Builds the *adverse* dataset** — starts from the full DrugBank approved-drug DDI pair list and
   removes pairs whose description indicates a purely beneficial ("therapeutic efficacy increased")
   interaction, leaving only adverse/negative-outcome interactions.
2. **Builds the *negative (non-interacting)* dataset** — randomly samples drug pairs that have **no**
   documented interaction at all, class-balanced against the adverse dataset.
3. **Enriches every drug** referenced by either dataset with core identity features pulled from the
   DrugBank XML dump: SMILES, ATC codes, target UniProt IDs, target FASTA sequences, and free-text
   target/gene identifiers.
4. **Backfills missing features** in two passes using PubChem, ChEMBL, UniProt, and the WHO ATC/DDD
   index, since DrugBank alone does not annotate every small molecule completely.
5. **Filters both pair datasets down** to only pairs where *both* drugs have a complete core feature
   set (SMILES + ATC + targets), then saves the result as Parquet.

### Data lineage (where the raw inputs came from)
| Output of this notebook | Built from | Original exploratory notebook |
|---|---|---|
| Adverse-only pairs (`is_therapeutic_efficacy == False`) | Full DrugBank approved-drug DDI pair export (`drugbank_approved_small_2369_1129743_ddi_pairs.csv`) | `notebooks_test/non_adverse_ddi_extraction.ipynb` |
| Negative (non-interacting) samples | Adverse-only pairs above (used to build the "documented pairs" exclusion set) | `notebooks_test/negative_ddi_sampling.ipynb` |
| Per-drug SMILES / ATC / targets | DrugBank full XML dump (`drugbank_full_database_V5.1.14.zip`) | `notebooks_test/weeding_out_unusable_unapproved_drugs.ipynb` (draft precursor of this notebook) |
| Backfilled SMILES / ATC / targets | PubChem, ChEMBL, UniProt, WHO ATC/DDD index (live REST/HTML calls) | same as above |

The full 1.1M-row raw DDI export and the DrugBank XML dump are **not checked into this git repo** (too1
large) — they live on the external data drive referenced by the `RAW_*` paths in the configuration cell
below. If those files aren't available in your environment, skip Sections 1–2 (leave the rebuild flags
`False`) and this notebook will load the already-generated `negative_ddi_samples.csv` /
`*_positive_removed.csv` files instead.

### Relationship to `notebooks/h1_biological_overlap/biological_overlap.ipynb`
An earlier version of this notebook (`data_curation_notebook.ipynb`) also tried to backfill pathway
neighbor proteins, GO terms, and Pfam domains per drug (its "Cell 9f–9i"). **That logic is superseded**
by `biological_overlap.ipynb`, which rebuilds all of that — and more — directly from the DrugBank XML +
ChEMBL in a single, correct pass via `smpdb_protein_pathway.parse_drugbank_biological_profiles()`:
- The `L0 ⊆ L1 ⊆ L2` protein-role ladder (`L0 = T(d)` direct targets, `L1 = L0 ∪ enzymes ∪
  transporters`, `L2 = L1 ∪ carriers`), pooling DrugBank targets **and** ChEMBL mechanism/bioactivity
  targets for `T(d)`.
- Gene Ontology terms split by aspect (`GO_MF`, `GO_BP`, `GO_CC`) instead of one flat list.
- Native SMPDB pathway membership (`Φ_native`) *and* an inferred two-hop pathway-neighbor expansion
  (`Φ_infer`), reseeded from the pooled `L0`.
- Jaccard similarity and directional Tversky containment for every one of the layers above.

Because of that, **this notebook deliberately stops at the "complete core feature" gate** (SMILES + ATC
+ direct targets) and does not attempt pathway/GO/Pfam enrichment — that work belongs in
`biological_overlap.ipynb`, which consumes the `adverse_final_df` / `negative_final_df` outputs produced
here.

```mermaid
flowchart TD
    A[Full DrugBank DDI pair export] -->|remove therapeutic-efficacy pairs| B[Adverse-only pairs]
    B -->|random sample, class-balanced, no documented interaction| C[Negative pairs]
    B --> D[Section 3: load + infer pair columns]
    C --> D
    D --> E[Section 5: parse DrugBank XML for needed drugs]
    E --> F[Section 6: restrict to small-molecule, non-withdrawn drugs]
    F --> G[Section 7: attach features + completeness flags]
    G --> H[Section 8-10: backfill via PubChem / ChEMBL / UniProt / WHO ATC]
    H --> I[Section 11: final completeness gate + save Parquet]
    I --> J[biological_overlap.ipynb: L0/L1/L2, GO aspects, native + inferred pathways]
```


In [1]:
# ============================================================================
# Imports & global configuration
# ============================================================================
# Every path used anywhere in this notebook is declared here so the whole
# pipeline can be re-pointed at a different machine/drive by editing one cell.

import os
import re
import time
import zipfile
import random
from pathlib import Path
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests

CONFIG = {
    # --- Rebuild toggles -----------------------------------------------------
    # Both default to False because they require the full 1.1M-row raw DDI
    # export and take several minutes to (re)generate. Flip to True only when
    # the upstream raw file actually changed and these datasets need refreshing.
    "rebuild_adverse_from_raw": False,
    "rebuild_negative_from_raw": False,

    # --- Raw inputs (tracked locally in this repo under data/) --------------
    "raw_all_ddi_pairs_path": r"C:\Users\ashto\ddi-prediction\data\sample\drugbank_approved_small_2369_1129743_ddi_pairs.csv",
    "drugbank_xml_path": r"C:\Users\ashto\ddi-prediction\data\raw\drugbank_full_database_V5.1.14.zip",

    # --- Adverse / negative dataset locations --------------------------------
    # These are the two files this notebook previously called
    # `original_adverse_dataset` / `original_negative_dataset`.
    "adverse_positive_removed_path": r"C:\Users\ashto\ddi-prediction\data\sample\drugbank_approved_small_1113772_ddi_pairs_positive_removed.csv",
    "positive_conservative_path": r"C:\Users\ashto\ddi-prediction\data\sample\positive_non_adverse_ddis_conservative.csv",
    "negative_samples_path": r"C:\Users\ashto\ddi-prediction\data\sample\negative_ddi_samples.csv",

    # --- Final curated outputs (tracked locally in this repo) ----------------
    "output_dir": r"C:\Users\ashto\ddi-prediction\data\sample",
}

print("Configuration loaded.")
print(f"Rebuild adverse dataset from raw : {CONFIG['rebuild_adverse_from_raw']}")
print(f"Rebuild negative dataset from raw: {CONFIG['rebuild_negative_from_raw']}")


Configuration loaded.
Rebuild adverse dataset from raw : False
Rebuild negative dataset from raw: False


## Section 1 — Build the adverse-only dataset

**Ported from:** `notebooks_test/non_adverse_ddi_extraction.ipynb` (see also
`data/sample/EXTRACTION_PROCESS.md` for the full write-up of this methodology).

### Problem
The raw DrugBank export contains **every** documented interaction between the 2,369 approved drugs
(1,129,743 pairs) — including interactions that are explicitly *beneficial*
(e.g. "The therapeutic efficacy of Bivalirudin can be increased when used in combination with
Quinine."). Training an "adverse DDI" classifier on this raw set would contaminate the negative-outcome
class with positive-outcome examples.

### Approach: conservative pattern matching
Three classification strategies were evaluated (see `EXTRACTION_PROCESS.md` for the full comparison):

| Approach | Pattern scope | Positive DDIs found | Verdict |
|---|---|---|---|
| Original | 6 named activities (analgesic, bronchodilatory, ...) | 22,257 | too restrictive |
| Expanded | *any* "may increase/decrease the X activity of" | 122,138 | contaminated — many of these activities (e.g. anticoagulant, analgesic) have **hundreds of times more** adverse-risk mentions elsewhere in the dataset, so "activity enhancement" alone doesn't reliably mean "beneficial" |
| **Conservative (used here)** | only `"therapeutic efficacy of ... can be increased"` | 15,971 | clinically unambiguous — this phrasing is DrugBank's own language for "this combination improves treatment outcomes" |

We use the **conservative** pattern only. Everything that doesn't match it is kept as "adverse."

### Output
- `positive_non_adverse_ddis_conservative.csv` — the 15,971 beneficial pairs (kept for reference /
  potential positive-interaction modeling, not used further in this notebook).
- The **adverse-only** dataset (1,129,743 − 15,971 = 1,113,772 pairs) — this becomes
  `original_adverse_dataset` for the rest of the pipeline.


In [2]:
# Conservative "therapeutic efficacy increased" pattern — the only signal we trust
# as unambiguously beneficial (see markdown above for why the broader patterns were rejected).
pattern_therapeutic_efficacy = re.compile(
    r"therapeutic efficacy of .* can be increased",
    re.IGNORECASE,
)


def _detect_description_column(df):
    """Find whichever column holds the free-text interaction description."""
    for col in df.columns:
        if "description" in col.lower() or "mechanism" in col.lower() or "effect" in col.lower():
            return col
    raise ValueError(f"Could not auto-detect a description column. Columns: {df.columns.tolist()}")


if CONFIG["rebuild_adverse_from_raw"]:
    print("Loading full raw DDI export (this is the 1.1M-row file, may take a moment)...")
    raw_all_pairs_df = pd.read_csv(CONFIG["raw_all_ddi_pairs_path"])
    print(f"Loaded {len(raw_all_pairs_df):,} raw DDI pairs | columns: {raw_all_pairs_df.columns.tolist()}")

    desc_col = _detect_description_column(raw_all_pairs_df)
    print(f"Using description column: {desc_col!r}")

    raw_all_pairs_df["is_therapeutic_efficacy"] = (
        raw_all_pairs_df[desc_col]
        .apply(lambda d: bool(pattern_therapeutic_efficacy.search(str(d))) if pd.notna(d) else False)
    )

    positive_conservative_df = raw_all_pairs_df.loc[raw_all_pairs_df["is_therapeutic_efficacy"]].copy()
    adverse_positive_removed_df = raw_all_pairs_df.loc[~raw_all_pairs_df["is_therapeutic_efficacy"]].copy()

    print(f"\nTherapeutic-efficacy (beneficial) pairs : {len(positive_conservative_df):,}")
    print(f"Adverse-only pairs (kept)                : {len(adverse_positive_removed_df):,}")

    positive_conservative_df.to_csv(CONFIG["positive_conservative_path"], index=False)
    adverse_positive_removed_df.to_csv(CONFIG["adverse_positive_removed_path"], index=False)
    print("\nSaved both outputs.")
else:
    print("Skipping rebuild (CONFIG['rebuild_adverse_from_raw'] is False).")
    print(f"Loading previously-built adverse-only dataset from:\n  {CONFIG['adverse_positive_removed_path']}")
    adverse_positive_removed_df = pd.read_csv(CONFIG["adverse_positive_removed_path"])
    print(f"Loaded {len(adverse_positive_removed_df):,} adverse-only pairs")


Loading full raw DDI export (this is the 1.1M-row file, may take a moment)...
Loaded 1,129,743 raw DDI pairs | columns: ['drug1_id', 'drug1_name', 'drug2_id', 'drug2_name', 'description', 'pair_key']
Using description column: 'description'

Therapeutic-efficacy (beneficial) pairs : 15,971
Adverse-only pairs (kept)                : 1,113,772

Saved both outputs.


## Section 2 — Build the negative (non-interacting) dataset

**Ported from:** `notebooks_test/negative_ddi_sampling.ipynb`

### Problem
A DDI classifier needs a "no interaction" class, but DrugBank only documents pairs that **do**
interact — there's no ready-made list of pairs that don't. We have to construct one.

### Approach
1. Take every drug that appears in the adverse-only dataset built in Section 1.
2. Build the set of every `{drug_a, drug_b}` pair that DrugBank *does* document (in either direction —
   direction doesn't matter for "is there a documented interaction at all").
3. Randomly sample pairs from the full combinatorial space of those drugs, keeping only pairs that are
   **not** in the documented set, until we have as many negative pairs as there are adverse pairs
   (1:1 class balance).
4. Validate: no duplicates, no accidental overlap with the documented set.

`random.seed(42)` is fixed for reproducibility — rerunning this cell regenerates the exact same negative
sample.


In [3]:
if CONFIG["rebuild_negative_from_raw"]:
    print(f"Building negative samples from {len(adverse_positive_removed_df):,} adverse-only pairs...")

    d1_col, d2_col = "drug1_id", "drug2_id"
    name1_col, name2_col = "drug1_name", "drug2_name"

    # Every drug that appears anywhere in the adverse dataset.
    all_drugs = sorted(set(adverse_positive_removed_df[d1_col]) | set(adverse_positive_removed_df[d2_col]))
    drug_name_lookup = {}
    for row in adverse_positive_removed_df.itertuples(index=False):
        drug_name_lookup[getattr(row, d1_col)] = getattr(row, name1_col)
        drug_name_lookup[getattr(row, d2_col)] = getattr(row, name2_col)
    print(f"Unique drugs: {len(all_drugs):,} | Names mapped: {len(drug_name_lookup):,}")

    # Every pair DrugBank documents as interacting, direction-agnostic.
    documented_pairs = set(
        tuple(sorted(pair))
        for pair in zip(adverse_positive_removed_df[d1_col], adverse_positive_removed_df[d2_col])
    )
    total_possible_pairs = len(list(combinations(all_drugs, 2)))
    print(f"Documented adverse pairs   : {len(documented_pairs):,}")
    print(f"Total possible drug pairs  : {total_possible_pairs:,}")
    print(f"Available negative pairs   : {total_possible_pairs - len(documented_pairs):,}")

    # Class-balance the negative sample against the adverse dataset size.
    target_negative_samples = len(documented_pairs)
    random.seed(42)  # reproducibility
    negative_pairs_seen = set()  # tracks pairs already accepted this run, so we never sample the same pair twice
    attempts, max_attempts = 0, target_negative_samples * 20

    print(f"\nSampling {target_negative_samples:,} non-documented pairs...")
    while len(negative_pairs_seen) < target_negative_samples and attempts < max_attempts:
        pair_key = tuple(sorted(random.sample(all_drugs, 2)))
        if pair_key not in documented_pairs and pair_key not in negative_pairs_seen:
            negative_pairs_seen.add(pair_key)
        attempts += 1

    negative_pairs = list(negative_pairs_seen)

    print(f"Generated {len(negative_pairs):,} negative pairs in {attempts:,} attempts")
    if len(negative_pairs) < target_negative_samples:
        print(f"WARNING: only reached {len(negative_pairs):,}/{target_negative_samples:,} — documented-pair density may be high")

    negative_samples_df = pd.DataFrame([
        {
            "drug1_id": a,
            "drug1_name": drug_name_lookup.get(a, "Unknown"),
            "drug2_id": b,
            "drug2_name": drug_name_lookup.get(b, "Unknown"),
            "description": "No documented interaction",
        }
        for a, b in negative_pairs
    ])

    # Validation
    assert negative_samples_df[["drug1_id", "drug2_id"]].isnull().sum().sum() == 0, "null drug ids found"
    assert len(negative_samples_df) == len(negative_samples_df.drop_duplicates(subset=["drug1_id", "drug2_id"])), "duplicate pairs found"
    overlap = sum(
        1 for a, b in zip(negative_samples_df["drug1_id"], negative_samples_df["drug2_id"])
        if tuple(sorted((a, b))) in documented_pairs
    )
    assert overlap == 0, f"{overlap} negative pairs overlap with documented adverse pairs"
    print("\nValidation passed: no nulls, no duplicates, no overlap with documented pairs.")

    negative_samples_df.to_csv(CONFIG["negative_samples_path"], index=False)
    print(f"Saved {len(negative_samples_df):,} negative samples to:\n  {CONFIG['negative_samples_path']}")
else:
    print("Skipping rebuild (CONFIG['rebuild_negative_from_raw'] is False).")
    print(f"Loading previously-built negative samples from:\n  {CONFIG['negative_samples_path']}")
    negative_samples_df = pd.read_csv(CONFIG["negative_samples_path"])
    print(f"Loaded {len(negative_samples_df):,} negative samples")


Building negative samples from 1,113,772 adverse-only pairs...
Unique drugs: 4,527 | Names mapped: 4,527
Documented adverse pairs   : 1,113,772
Total possible drug pairs  : 10,244,601
Available negative pairs   : 9,130,829

Sampling 1,113,772 non-documented pairs...
Generated 1,113,772 negative pairs in 1,333,568 attempts

Validation passed: no nulls, no duplicates, no overlap with documented pairs.
Saved 1,113,772 negative samples to:
  C:\Users\ashto\ddi-prediction\data\sample\negative_ddi_samples.csv


## Section 3 — Establish working copies & infer pair-id columns

From here on the pipeline is symmetric for both datasets, so we normalize to two variables:
`original_negative_dataset` and `original_adverse_dataset`. Column names for the two drug IDs in a pair
aren't perfectly consistent across every raw file this project has produced over time, so
`infer_pair_cols` checks a list of known naming conventions rather than hardcoding one.

We also compute `needed_ids` — the full union of DrugBank IDs referenced by either dataset. Every
downstream DrugBank XML parse and API backfill is scoped to just this set, which is what makes the rest
of the notebook tractable (the raw DrugBank export is >1 GB and describes ~17,000 drugs; we only care
about the ~thousands actually referenced here).


In [4]:
original_negative_dataset = negative_samples_df
original_adverse_dataset = adverse_positive_removed_df


def norm_id(x):
    if pd.isna(x):
        return np.nan
    return str(x).strip().upper()


def infer_pair_cols(df):
    """Locate the two drug-id columns in a pairs dataframe, tolerant of naming drift
    across the various raw files this project has accumulated."""
    candidates = [
        ("drug1", "drug2"),
        ("drug_1", "drug_2"),
        ("drugbank_id_1", "drugbank_id_2"),
        ("drug1_id", "drug2_id"),
        ("left_drugbank_id", "right_drugbank_id"),
        ("Drug1_ID", "Drug2_ID"),
    ]
    cols = {c.lower(): c for c in df.columns}
    for a, b in candidates:
        if a.lower() in cols and b.lower() in cols:
            return cols[a.lower()], cols[b.lower()]
    raise ValueError(f"Could not infer pair columns. Available: {list(df.columns)}")


neg_d1, neg_d2 = infer_pair_cols(original_negative_dataset)
adv_d1, adv_d2 = infer_pair_cols(original_adverse_dataset)

needed_ids = set(
    pd.concat([
        original_negative_dataset[neg_d1], original_negative_dataset[neg_d2],
        original_adverse_dataset[adv_d1], original_adverse_dataset[adv_d2],
    ], axis=0).dropna().map(norm_id).unique()
)

print(f"Negative pair cols: {neg_d1}, {neg_d2} | rows: {len(original_negative_dataset):,}")
print(f"Adverse pair cols : {adv_d1}, {adv_d2} | rows: {len(original_adverse_dataset):,}")
print(f"Unique DrugBank IDs needed: {len(needed_ids):,}")


Negative pair cols: drug1_id, drug2_id | rows: 1,113,772
Adverse pair cols : drug1_id, drug2_id | rows: 1,113,772
Unique DrugBank IDs needed: 4,527


## Section 4 — DrugBank XML parsing helpers

DrugBank's XML schema nests each drug's identity features in predictable but verbose sub-trees. These
helpers extract exactly the fields this pipeline needs, given a single `<drug>` element:

- **`_extract_smiles`** — SMILES string, preferring `calculated-properties` and falling back to
  `experimental-properties` if the calculated value is absent.
- **`_extract_atc_list`** — every ATC code attached to the drug (a drug can have more than one).
- **`_extract_targets`** — for every `<target>` entry: its UniProt accession (if sourced from
  Swiss-Prot/UniProtKB), amino-acid FASTA sequence, gene/protein/target *names* (kept separately as
  `other_ids` since they aren't stable database identifiers), and documented pharmacological actions.
- **`_extract_groups`** — DrugBank's approval-status tags (`approved`, `withdrawn`, `investigational`,
  ...), used in Section 6 to decide which drugs are eligible at all.

These are intentionally scoped to *targets only* (not enzymes/transporters/carriers) — the broader
`L1`/`L2` protein-role ladder is `biological_overlap.ipynb`'s responsibility, not this notebook's.


In [5]:
def _extract_smiles(elem, ns):
    calc_props = elem.find(f"{ns}calculated-properties")
    if calc_props is not None:
        for prop in calc_props.findall(f"{ns}property"):
            kind_el = prop.find(f"{ns}kind")
            if kind_el is not None and (kind_el.text or "").strip().upper() == "SMILES":
                val_el = prop.find(f"{ns}value")
                if val_el is not None and val_el.text:
                    return val_el.text.strip()
    exp_props = elem.find(f"{ns}experimental-properties")
    if exp_props is not None:
        for prop in exp_props.findall(f"{ns}property"):
            kind_el = prop.find(f"{ns}kind")
            if kind_el is not None and (kind_el.text or "").strip().upper() == "SMILES":
                val_el = prop.find(f"{ns}value")
                if val_el is not None and val_el.text:
                    return val_el.text.strip()
    return np.nan


def _extract_atc_list(elem, ns):
    codes = set()
    atc_block = elem.find(f"{ns}atc-codes")
    if atc_block is not None:
        for a in atc_block.findall(f"{ns}atc-code"):
            code = a.get("code")
            if code:
                codes.add(code)
    return sorted(codes)


def _extract_targets(elem, ns):
    """Returns dict with: uniprot_ids, fasta_sequences, other_ids (gene/protein/target
    names, non-UniProt external ids), actions (documented pharmacological actions)."""
    uniprot_ids, fasta_sequences, other_ids, actions = set(), set(), set(), set()

    targets_block = elem.find(f"{ns}targets")
    if targets_block is None:
        return {"uniprot_ids": [], "fasta_sequences": [], "other_ids": [], "actions": []}

    for t in targets_block.findall(f"{ns}target"):
        t_name_el = t.find(f"{ns}name")
        if t_name_el is not None and t_name_el.text:
            other_ids.add(t_name_el.text.strip())

        actions_block = t.find(f"{ns}actions")
        if actions_block is not None:
            for act in actions_block.findall(f"{ns}action"):
                if act.text:
                    actions.add(act.text.strip())

        polypeptide_block = t.find(f"{ns}polypeptide")
        if polypeptide_block is not None:
            pid = (polypeptide_block.get("id") or "").strip()
            source = (polypeptide_block.get("source") or "").strip().lower()

            if pid:
                if source in ("swiss-prot", "uniprotkb", ""):
                    uniprot_ids.add(pid)
                else:
                    other_ids.add(pid)

            gene_el = polypeptide_block.find(f"{ns}gene-name")
            if gene_el is not None and gene_el.text:
                other_ids.add(gene_el.text.strip())

            pname_el = polypeptide_block.find(f"{ns}name")
            if pname_el is not None and pname_el.text:
                other_ids.add(pname_el.text.strip())

            seq_el = polypeptide_block.find(f"{ns}amino-acid-sequence")
            if seq_el is not None and seq_el.text:
                lines = seq_el.text.strip().splitlines()
                seq_body = "".join(lines[1:]) if lines and lines[0].startswith(">") else "".join(lines)
                if seq_body:
                    fasta_sequences.add(seq_body.strip())

            ext_block = polypeptide_block.find(f"{ns}external-identifiers")
            if ext_block is not None:
                for ext in ext_block.findall(f"{ns}external-identifier"):
                    res_el = ext.find(f"{ns}resource")
                    id_el = ext.find(f"{ns}identifier")
                    res = (res_el.text or "").strip().lower() if res_el is not None else ""
                    ident = (id_el.text or "").strip() if id_el is not None else ""
                    if not ident:
                        continue
                    if res == "uniprotkb":
                        uniprot_ids.add(ident)
                    else:
                        other_ids.add(ident)

    return {
        "uniprot_ids": sorted(uniprot_ids),
        "fasta_sequences": sorted(fasta_sequences),
        "other_ids": sorted(other_ids),
        "actions": sorted(actions),
    }


def _extract_groups(elem, ns):
    groups = []
    g_block = elem.find(f"{ns}groups")
    if g_block is not None:
        for g in g_block.findall(f"{ns}group"):
            if g.text:
                groups.append(g.text.strip().lower())
    return sorted(set(groups))


print("DrugBank XML parsing helpers defined.")


DrugBank XML parsing helpers defined.


## Section 5 — Stream-parse the DrugBank XML for the needed drugs

The full DrugBank dump is a single XML file well over a gigabyte, so we use `ET.iterparse` to stream
through it `<drug>` element by `<drug>` element rather than loading the whole tree into memory, calling
`elem.clear()` after each one to free it immediately.

**Inclusion filter:** `type == "small molecule"` and `"withdrawn" not in groups`. This deliberately
includes both **approved** and **unapproved** small molecules (as long as they aren't withdrawn) — the
downstream feature-completeness gate in Section 7 is the real filter; here we just avoid wasting time
parsing biologics, and we track `is_approved` separately so approved/unapproved counts can be reported
at every later stage.


In [6]:
def _parse_drugbank_stream(stream, wanted_ids):
    records = []
    total_drugs = 0
    matched = 0

    for _, elem in ET.iterparse(stream, events=("end",)):
        if not elem.tag.endswith("drug"):
            continue

        ns = elem.tag.split("}")[0] + "}" if "}" in elem.tag else ""
        total_drugs += 1

        dbid = None
        for x in elem.findall(f"{ns}drugbank-id"):
            if x.get("primary") == "true":
                dbid = (x.text or "").strip().upper()
                break
        if dbid is None:
            first = elem.find(f"{ns}drugbank-id")
            if first is not None and first.text:
                dbid = first.text.strip().upper()

        if (dbid is None) or (dbid not in wanted_ids):
            elem.clear()
            continue

        name_el = elem.find(f"{ns}name")
        drug_name = name_el.text.strip() if name_el is not None and name_el.text else None

        groups = [g.text for g in elem.findall(f"{ns}groups/{ns}group") if g.text]

        # Inclusion filter: small molecule + not withdrawn (approval NOT required here).
        if not (elem.get("type") == "small molecule" and "withdrawn" not in groups):
            elem.clear()
            continue

        drug_type = (elem.get("type") or "").strip().lower()
        is_small_molecule = drug_type == "small molecule"
        is_approved = "approved" in groups
        is_withdrawn = "withdrawn" in groups

        atc_list = _extract_atc_list(elem, ns)
        targets = _extract_targets(elem, ns)

        records.append({
            "drugbank_id": dbid,
            "drug_name": drug_name,
            "drug_type": drug_type,
            "is_small_molecule": is_small_molecule,
            "smiles": _extract_smiles(elem, ns),
            "atc_codes_list": atc_list,
            "target_uniprot_ids": targets["uniprot_ids"],
            "target_fasta_sequences": targets["fasta_sequences"],
            "target_other_ids": targets["other_ids"],
            "target_actions": targets["actions"],
            "drug_groups": groups,
            "is_approved": bool(is_approved),
            "is_withdrawn": bool(is_withdrawn),
        })

        matched += 1
        if matched % 200 == 0:
            print(f"Inspected {total_drugs} | Matched {matched}")

        elem.clear()

    print(f"\nFinished. Total inspected: {total_drugs} | Matched: {matched}")
    return pd.DataFrame(records).drop_duplicates(subset=["drugbank_id"])


def parse_drugbank_full(xml_or_zip_path, wanted_ids):
    p = Path(xml_or_zip_path)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p, "r") as zf:
            xml_members = [n for n in zf.namelist() if n.lower().endswith(".xml")]
            if not xml_members:
                raise FileNotFoundError("No XML found inside zip.")
            print("Streaming:", xml_members[0])
            with zf.open(xml_members[0], "r") as f:
                return _parse_drugbank_stream(f, wanted_ids)
    else:
        with open(p, "rb") as f:
            return _parse_drugbank_stream(f, wanted_ids)


drugbank_df = parse_drugbank_full(CONFIG["drugbank_xml_path"], needed_ids)
print("\ndrugbank_df shape:", drugbank_df.shape)
print("is_small_molecule counts:\n", drugbank_df["is_small_molecule"].value_counts())
print("is_approved counts:\n", drugbank_df["is_approved"].value_counts())
drugbank_df.head()


Streaming: drugbank_full_database_V5.1.14.xml
Inspected 281054 | Matched 200
Inspected 282376 | Matched 400
Inspected 284766 | Matched 600
Inspected 286791 | Matched 800
Inspected 296206 | Matched 1000
Inspected 641017 | Matched 1200
Inspected 995669 | Matched 1400
Inspected 996449 | Matched 1600
Inspected 1002519 | Matched 1800
Inspected 1003708 | Matched 2000
Inspected 1004281 | Matched 2200
Inspected 1004937 | Matched 2400
Inspected 1005774 | Matched 2600
Inspected 1006154 | Matched 2800
Inspected 1006560 | Matched 3000
Inspected 1007859 | Matched 3200

Finished. Total inspected: 1014328 | Matched: 3316

drugbank_df shape: (3316, 13)
is_small_molecule counts:
 is_small_molecule
True    3316
Name: count, dtype: int64
is_approved counts:
 is_approved
True     1899
False    1417
Name: count, dtype: int64


,drugbank_id,drug_name,drug_type,is_small_molecule,smiles,atc_codes_list,target_uniprot_ids,target_fasta_sequences,target_other_ids,target_actions,drug_groups,is_approved,is_withdrawn
0,DB00006,Bivalirudin,small molecule,True,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,[B01AE06],[P00734],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...",[inhibitor],"[approved, investigational]",True,False
1,DB00014,Goserelin,small molecule,True,CC(C)C[C@H](NC(=O)[C@@H](COC(C)(C)C)NC(=O)[C@H...,[L02AE03],"[P01148, P22888, P30968]",[MANSASPEQNQNHCSAINNSIPLMQGNLPTLTLSGKIRVTVTFFL...,"[183422, 254, 256, 903746, GNRH1, GNRHR, GNRHR...","[activator, agonist]","[approved, investigational]",True,False
2,DB00027,Gramicidin D,small molecule,True,CC(C)C[C@@H](NC(=O)CNC(=O)[C@@H](NC=O)C(C)C)C(...,[R02AB30],[P0AC13],[MKLFAQGTSLDLSHPHVMGILNVTPDSFSDGGTHNSLIDAVKHAN...,"[41273, DHPS_ECOLI, Dihydropteroate synthase, ...",[binder],"[approved, investigational]",True,False
3,DB00035,Desmopressin,small molecule,True,NC(=O)CC[C@@H]1NC(=O)[C@H](CC2=CC=CC=C2)NC(=O)...,[H01BA02],"[P30518, P30559, P37288, P47901]",[MDSGPLWDANPTPRGTLSAPNATTPWLGRDEELAKVEIGVLATVL...,"[28418, 34765, 366, 367, 368, 369, 563982, 667...",[agonist],"[approved, investigational]",True,False
4,DB00080,Daptomycin,small molecule,True,CCCCCCCCCC(=O)N[C@@H](CC1=CNC2=C1C=CC=C2)C(=O)...,[J01XX09],[P0AC13],[MKLFAQGTSLDLSHPHVMGILNVTPDSFSDGGTHNSLIDAVKHAN...,"[41273, Cytoplasmic membrane, DHPS_ECOLI, Dihy...","[binder, incorporation into and destabilization]","[approved, investigational]",True,False


## Section 6 — Restrict both pair datasets to eligible small-molecule drugs

A pair is only usable if **both** drugs in it are small molecules we successfully parsed and that
aren't withdrawn (`sm_ids`). Any pair referencing a drug outside that set (biologics, drugs missing from
the XML entirely, withdrawn drugs) is dropped here, before we spend any time on feature backfill.


In [7]:
sm_ids = set(
    drugbank_df.loc[
        (drugbank_df["is_small_molecule"] == True) & (drugbank_df["is_withdrawn"] == False),
        "drugbank_id",
    ]
)
print(f"Small-molecule (approved + unapproved, non-withdrawn) drugs available: {len(sm_ids):,}")


def filter_pairs_to_ids(df, d1_col, d2_col, allowed_ids):
    out = df.copy()
    d1 = out[d1_col].astype(str).str.strip().str.upper()
    d2 = out[d2_col].astype(str).str.strip().str.upper()
    mask = d1.isin(allowed_ids) & d2.isin(allowed_ids)
    return out.loc[mask].copy()


negative_df = filter_pairs_to_ids(original_negative_dataset, neg_d1, neg_d2, sm_ids)
adverse_df = filter_pairs_to_ids(original_adverse_dataset, adv_d1, adv_d2, sm_ids)

print(f"Negative: {len(original_negative_dataset):,} -> {len(negative_df):,}")
print(f"Adverse : {len(original_adverse_dataset):,} -> {len(adverse_df):,}")


Small-molecule (approved + unapproved, non-withdrawn) drugs available: 3,316
Negative: 1,113,772 -> 570,462
Adverse : 1,113,772 -> 818,997


## Section 7 — Attach per-drug features to pairs & build the completeness table

Two things happen here:
1. `attach_features` joins each pair row to its two drugs' `drugbank_df` features (suffixed `_1`/`_2`),
   giving pair-level dataframes with columns like `smiles_1`, `smiles_2`, `atc_codes_list_1`, etc.
2. We separately build `per_drug_df` — **one row per unique drug** (not per pair) restricted to only
   the drugs actually referenced by our filtered pair datasets (`relevant_ids`) — and compute boolean
   completeness flags:
   - `has_smiles` — non-null SMILES string
   - `has_atc` — at least one ATC code
   - `has_targets` — at least one UniProt ID *or* other target identifier
   - `has_all_features` — all three of the above

`per_drug_df` is the table Sections 8–10 iteratively backfill; pairs are only re-attached from it once
at the very end (Section 11).


In [8]:
feat_lookup = drugbank_df.set_index("drugbank_id")
feature_cols = ["smiles", "atc_codes_list", "target_uniprot_ids", "target_fasta_sequences", "target_other_ids"]


def attach_features(df, d1_col, d2_col, lookup_df, cols):
    out = df.copy()
    out["_d1"] = out[d1_col].astype(str).str.strip().str.upper()
    out["_d2"] = out[d2_col].astype(str).str.strip().str.upper()
    for col in cols:
        col_map = lookup_df[col].to_dict()
        out[f"{col}_1"] = out["_d1"].map(col_map)
        out[f"{col}_2"] = out["_d2"].map(col_map)
    return out.drop(columns=["_d1", "_d2"])


negative_feat_df = attach_features(negative_df, neg_d1, neg_d2, feat_lookup, feature_cols)
adverse_feat_df = attach_features(adverse_df, adv_d1, adv_d2, feat_lookup, feature_cols)

print("Negative feat shape:", negative_feat_df.shape)
print("Adverse  feat shape:", adverse_feat_df.shape)

# --- Per-drug completeness table -------------------------------------------


def list_is_empty(x):
    if isinstance(x, list):
        return len(x) == 0
    return pd.isna(x)


drugbank_df["has_smiles"] = drugbank_df["smiles"].notna()
drugbank_df["has_atc"] = drugbank_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
drugbank_df["has_targets"] = (
    drugbank_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x))
    | drugbank_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
drugbank_df["has_all_features"] = drugbank_df["has_smiles"] & drugbank_df["has_atc"] & drugbank_df["has_targets"]

relevant_ids = set(
    pd.concat([
        negative_df[neg_d1], negative_df[neg_d2],
        adverse_df[adv_d1], adverse_df[adv_d2],
    ]).astype(str).str.strip().str.upper().unique()
)

per_drug_df = drugbank_df.loc[drugbank_df["drugbank_id"].isin(relevant_ids)].copy()

print(f"\nRelevant unique drugs: {len(relevant_ids):,}")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

missing_smiles_ids = per_drug_df.loc[~per_drug_df["has_smiles"], "drugbank_id"].tolist()
missing_atc_ids = per_drug_df.loc[~per_drug_df["has_atc"], "drugbank_id"].tolist()
missing_targets_ids = per_drug_df.loc[~per_drug_df["has_targets"], "drugbank_id"].tolist()

print(f"\nMissing SMILES : {len(missing_smiles_ids):,}")
print(f"Missing ATC    : {len(missing_atc_ids):,}")
print(f"Missing targets: {len(missing_targets_ids):,}")


Negative feat shape: (570462, 15)
Adverse  feat shape: (818997, 17)

Relevant unique drugs: 3,316
has_smiles          0.972557
has_atc             0.661339
has_targets         0.749698
has_all_features    0.509047
dtype: float64

Missing SMILES : 91
Missing ATC    : 1,123
Missing targets: 830


## Section 8 — Backfill pass 1: PubChem (SMILES) + ChEMBL (ATC & targets)

DrugBank's own annotations are incomplete for a meaningful fraction of drugs (especially unapproved
ones). Rather than discard them immediately, we try two more authoritative sources first:

- **PubChem** — looked up first by DrugBank ID via PubChem's cross-reference (`xref`) endpoint, falling
  back to a name-based lookup if that fails. Used only for missing SMILES.
- **ChEMBL** — molecule ID resolved by drug name, then queried for `atc_classifications` (ATC codes)
  and `mechanism`-endpoint target ChEMBL IDs. Used for missing ATC codes and missing targets.

Both backfills run through a `ThreadPoolExecutor` since these are network-bound REST calls — I/O
parallelism gives a large speedup here without any extra infrastructure.


In [9]:
# --- PubChem: backfill missing SMILES ---------------------------------------

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"


def pubchem_smiles_by_drugbank_xref(drugbank_id, timeout=10):
    url = f"{PUBCHEM_BASE}/compound/xref/RegistryID/{drugbank_id}/property/CanonicalSMILES/JSON"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            props = r.json().get("PropertyTable", {}).get("Properties", [])
            if props:
                return props[0].get("CanonicalSMILES")
    except Exception:
        pass
    return None


def pubchem_smiles_by_name(drug_name, timeout=10):
    if not drug_name:
        return None
    url = f"{PUBCHEM_BASE}/compound/name/{requests.utils.quote(drug_name)}/property/CanonicalSMILES/JSON"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            props = r.json().get("PropertyTable", {}).get("Properties", [])
            if props:
                return props[0].get("CanonicalSMILES")
    except Exception:
        pass
    return None


pubchem_smiles_results = {}
for row in per_drug_df.loc[~per_drug_df["has_smiles"]].itertuples():
    dbid, name = row.drugbank_id, row.drug_name
    smi = pubchem_smiles_by_drugbank_xref(dbid) or pubchem_smiles_by_name(name)
    pubchem_smiles_results[dbid] = smi
    time.sleep(0.2)  # be polite to PubChem's rate limits
    if smi:
        print(f"[PubChem] {dbid} ({name}) -> found SMILES")

resolved_smiles_count = sum(1 for v in pubchem_smiles_results.values() if v)
print(f"\nPubChem resolved {resolved_smiles_count}/{len(pubchem_smiles_results)} missing SMILES")

# --- ChEMBL: backfill missing ATC codes + targets (parallelized) ------------

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
CHEMBL_MAX_WORKERS = 8


def chembl_molecule_id_by_name(drug_name, timeout=10):
    if not drug_name:
        return None
    url = f"{CHEMBL_BASE}/molecule/search.json"
    try:
        r = requests.get(url, params={"q": drug_name}, timeout=timeout)
        if r.status_code == 200:
            mols = r.json().get("molecules", [])
            if mols:
                return mols[0].get("molecule_chembl_id")
    except Exception:
        pass
    return None


def chembl_atc_codes(chembl_id, timeout=10):
    if not chembl_id:
        return []
    url = f"{CHEMBL_BASE}/molecule/{chembl_id}.json"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            return sorted(set(r.json().get("atc_classifications", []) or []))
    except Exception:
        pass
    return []


def chembl_targets(chembl_id, timeout=10):
    """Target ChEMBL IDs via the mechanism endpoint (drugs with a known mechanism of action)."""
    if not chembl_id:
        return []
    url = f"{CHEMBL_BASE}/mechanism.json"
    try:
        r = requests.get(url, params={"molecule_chembl_id": chembl_id}, timeout=timeout)
        if r.status_code == 200:
            targets = {m.get("target_chembl_id") for m in r.json().get("mechanisms", []) if m.get("target_chembl_id")}
            return sorted(targets)
    except Exception:
        pass
    return []


def _resolve_chembl(dbid, name, need_atc, need_targets):
    chembl_id = chembl_molecule_id_by_name(name)
    atc = chembl_atc_codes(chembl_id) if need_atc else None
    tgts = chembl_targets(chembl_id) if need_targets else None
    return dbid, atc, tgts


need_atc_or_targets = per_drug_df.loc[
    (~per_drug_df["has_atc"]) | (~per_drug_df["has_targets"]),
    ["drugbank_id", "drug_name", "has_atc", "has_targets"],
]

chembl_atc_results, chembl_target_results = {}, {}
print(f"\nResolving ATC/targets for {len(need_atc_or_targets)} drugs using {CHEMBL_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=CHEMBL_MAX_WORKERS) as executor:
    futures = {
        executor.submit(_resolve_chembl, row.drugbank_id, row.drug_name, not row.has_atc, not row.has_targets): row.drugbank_id
        for row in need_atc_or_targets.itertuples()
    }
    completed = 0
    for future in as_completed(futures):
        dbid, atc, tgts = future.result()
        if atc is not None:
            chembl_atc_results[dbid] = atc
        if tgts is not None:
            chembl_target_results[dbid] = tgts
        completed += 1
        if completed % 100 == 0:
            print(f"  ... {completed}/{len(need_atc_or_targets)} processed")

print(f"\nChEMBL resolved ATC for {sum(1 for v in chembl_atc_results.values() if v)}/{len(chembl_atc_results)}")
print(f"ChEMBL resolved targets for {sum(1 for v in chembl_target_results.values() if v)}/{len(chembl_target_results)}")



PubChem resolved 0/91 missing SMILES

Resolving ATC/targets for 1601 drugs using 8 parallel workers...
  ... 100/1601 processed
  ... 200/1601 processed
  ... 300/1601 processed
  ... 400/1601 processed
  ... 500/1601 processed
  ... 600/1601 processed
  ... 700/1601 processed
  ... 800/1601 processed
  ... 900/1601 processed
  ... 1000/1601 processed
  ... 1100/1601 processed
  ... 1200/1601 processed
  ... 1300/1601 processed
  ... 1400/1601 processed
  ... 1500/1601 processed
  ... 1600/1601 processed

ChEMBL resolved ATC for 60/1123
ChEMBL resolved targets for 99/830


In [10]:
# Merge pass-1 backfill results into per_drug_df and recompute completeness flags.
per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

for dbid, smi in pubchem_smiles_results.items():
    if smi:
        per_drug_df.loc[dbid, "smiles"] = smi

for dbid, atc in chembl_atc_results.items():
    if atc:
        existing = per_drug_df.at[dbid, "atc_codes_list"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "atc_codes_list"] = sorted(set(existing) | set(atc))

# Targets from ChEMBL are ChEMBL target IDs (not UniProt), so they're appended to
# `target_other_ids` rather than `target_uniprot_ids` to keep that column strictly UniProt.
for dbid, tgts in chembl_target_results.items():
    if tgts:
        existing = per_drug_df.at[dbid, "target_other_ids"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_other_ids"] = sorted(set(existing) | set(tgts))

per_drug_df = per_drug_df.reset_index(drop=True)

per_drug_df["has_smiles"] = per_drug_df["smiles"].notna()
per_drug_df["has_atc"] = per_drug_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_targets"] = (
    per_drug_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x))
    | per_drug_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
per_drug_df["has_all_features"] = per_drug_df["has_smiles"] & per_drug_df["has_atc"] & per_drug_df["has_targets"]

print("After PubChem + ChEMBL backfill (pass 1):")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())


After PubChem + ChEMBL backfill (pass 1):
has_smiles          0.972557
has_atc             0.679433
has_targets         0.779554
has_all_features    0.534982
dtype: float64


## Section 9 — Backfill pass 2: UniProt (targets) + WHO ATC/DDD index (ATC codes)

A second pass for whatever pass 1 still missed:

- **UniProt REST search** — queries reviewed (Swiss-Prot) entries by drug name and, failing that, by
  any target/gene names already scraped from DrugBank/ChEMBL (`target_other_ids`). Because these
  results genuinely are UniProt accessions, they're merged into `target_uniprot_ids` (not `other_ids`).
- **WHO ATC/DDD index** — the WHO index has no official public API, so this is a best-effort HTML
  scrape of the search results page, extracting codes with a regex
  (`[A-Z]\d{2}[A-Z]{2}\d{2}`). Kept to a conservative 4 parallel workers since the site publishes no
  rate-limit guidance.

Both are lower-confidence sources than pass 1, which is why they only run against whatever pass 1
didn't resolve.


In [11]:
# --- UniProt: second-pass target backfill -----------------------------------

UNIPROT_BASE = "https://rest.uniprot.org/uniprotkb/search"
UNIPROT_MAX_WORKERS = 8


def uniprot_search_by_name(query, timeout=10):
    """Search reviewed UniProtKB entries by free-text query (drug/gene/target name)."""
    if not query:
        return []
    params = {"query": f'"{query}" AND reviewed:true', "fields": "accession", "format": "json", "size": 5}
    try:
        r = requests.get(UNIPROT_BASE, params=params, timeout=timeout)
        if r.status_code == 200:
            return [res["primaryAccession"] for res in r.json().get("results", []) if "primaryAccession" in res]
    except Exception:
        pass
    return []


def _resolve_uniprot_targets(dbid, name, other_ids):
    found = set()
    candidates = [name] if name else []
    if isinstance(other_ids, list):
        candidates.extend(other_ids[:5])  # cap to avoid excessive querying per drug
    for cand in candidates:
        accs = uniprot_search_by_name(cand)
        found.update(accs)
        if found:
            break
    return dbid, sorted(found)


missing_targets_rows = per_drug_df.loc[~per_drug_df["has_targets"], ["drugbank_id", "drug_name", "target_other_ids"]]
uniprot_target_results = {}
print(f"Resolving targets via UniProt for {len(missing_targets_rows)} drugs using {UNIPROT_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=UNIPROT_MAX_WORKERS) as executor:
    futures = {
        executor.submit(_resolve_uniprot_targets, row.drugbank_id, row.drug_name, row.target_other_ids): row.drugbank_id
        for row in missing_targets_rows.itertuples()
    }
    for future in as_completed(futures):
        dbid, accs = future.result()
        uniprot_target_results[dbid] = accs

resolved_targets_count = sum(1 for v in uniprot_target_results.values() if v)
print(f"UniProt resolved targets for {resolved_targets_count}/{len(uniprot_target_results)} missing-target drugs")

# --- WHO ATC/DDD index: second-pass ATC backfill (best-effort HTML scrape) --

WHO_ATC_SEARCH_URL = "https://www.whocc.no/atc_ddd_index/"
WHO_MAX_WORKERS = 4  # conservative — WHO site publishes no rate-limit docs
ATC_CODE_PATTERN = re.compile(r"\b([A-Z]\d{2}[A-Z]{2}\d{2})\b")


def who_atc_lookup_by_name(drug_name, timeout=10):
    if not drug_name:
        return []
    try:
        r = requests.get(WHO_ATC_SEARCH_URL, params={"name": drug_name, "showdescription": "no"}, timeout=timeout)
        if r.status_code == 200:
            return sorted(set(ATC_CODE_PATTERN.findall(r.text)))
    except Exception:
        pass
    return []


missing_atc_rows = per_drug_df.loc[~per_drug_df["has_atc"], ["drugbank_id", "drug_name"]]
who_atc_results = {}
print(f"\nResolving ATC codes via WHO ATC/DDD Index for {len(missing_atc_rows)} drugs using {WHO_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=WHO_MAX_WORKERS) as executor:
    futures = {
        executor.submit(who_atc_lookup_by_name, row.drug_name): row.drugbank_id
        for row in missing_atc_rows.itertuples()
    }
    for future in as_completed(futures):
        dbid = futures[future]
        who_atc_results[dbid] = future.result()

resolved_who_atc_count = sum(1 for v in who_atc_results.values() if v)
print(f"WHO ATC resolved codes for {resolved_who_atc_count}/{len(who_atc_results)} missing-ATC drugs")


Resolving targets via UniProt for 731 drugs using 8 parallel workers...
UniProt resolved targets for 210/731 missing-target drugs

Resolving ATC codes via WHO ATC/DDD Index for 1063 drugs using 4 parallel workers...
WHO ATC resolved codes for 28/1063 missing-ATC drugs


In [12]:
# Merge pass-2 backfill results into per_drug_df and recompute completeness flags.
per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)

for dbid, accs in uniprot_target_results.items():
    if accs:
        existing = per_drug_df.at[dbid, "target_uniprot_ids"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "target_uniprot_ids"] = sorted(set(existing) | set(accs))

for dbid, codes in who_atc_results.items():
    if codes:
        existing = per_drug_df.at[dbid, "atc_codes_list"]
        existing = existing if isinstance(existing, list) else []
        per_drug_df.at[dbid, "atc_codes_list"] = sorted(set(existing) | set(codes))

per_drug_df = per_drug_df.reset_index(drop=True)

per_drug_df["has_smiles"] = per_drug_df["smiles"].notna()
per_drug_df["has_atc"] = per_drug_df["atc_codes_list"].apply(lambda x: not list_is_empty(x))
per_drug_df["has_targets"] = (
    per_drug_df["target_uniprot_ids"].apply(lambda x: not list_is_empty(x))
    | per_drug_df["target_other_ids"].apply(lambda x: not list_is_empty(x))
)
per_drug_df["has_all_features"] = per_drug_df["has_smiles"] & per_drug_df["has_atc"] & per_drug_df["has_targets"]

print("After UniProt + WHO ATC backfill (pass 2):")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_all_features"]].mean())

still_missing_smiles = per_drug_df.loc[~per_drug_df["has_smiles"], "drugbank_id"].tolist()
still_missing_atc = per_drug_df.loc[~per_drug_df["has_atc"], "drugbank_id"].tolist()
still_missing_targets = per_drug_df.loc[~per_drug_df["has_targets"], "drugbank_id"].tolist()

print(f"\nStill missing SMILES : {len(still_missing_smiles):,}")
print(f"Still missing ATC    : {len(still_missing_atc):,}")
print(f"Still missing targets: {len(still_missing_targets):,}")


After UniProt + WHO ATC backfill (pass 2):
has_smiles          0.972557
has_atc             0.687877
has_targets         0.842883
has_all_features    0.572979
dtype: float64

Still missing SMILES : 91
Still missing ATC    : 1,035
Still missing targets: 521


In [13]:
# Sanity check: are drugs actually keeping more than one ATC code after both backfill passes,
# or is something collapsing atc_codes_list down to a single value along the way?
atc_lengths = per_drug_df["atc_codes_list"].apply(lambda x: len(x) if isinstance(x, list) else 0)
print("Distribution of ATC-code-list lengths in per_drug_df at this point:")
print(atc_lengths.value_counts().sort_index())
print(f"\nDrugs with more than 1 ATC code: {(atc_lengths > 1).sum():,} / {len(per_drug_df):,}")

per_drug_df.loc[atc_lengths > 1, ["drugbank_id", "drug_name", "atc_codes_list"]].head(10)

Distribution of ATC-code-list lengths in per_drug_df at this point:
atc_codes_list
0     1035
1     1558
2      374
3      151
4       87
5       35
6       19
7       10
8       13
9        9
10       5
11       3
12       1
13       2
14       1
16       5
17       1
19       1
20       1
21       2
25       1
35       1
41       1
Name: count, dtype: int64

Drugs with more than 1 ATC code: 723 / 3,316


,drugbank_id,drug_name,atc_codes_list
5,DB00091,Cyclosporine,"[L04AD01, S01XA18]"
6,DB00115,Cyanocobalamin,"[B03BA01, B03BA51]"
10,DB00122,Choline,"[C10AB11, N02BA03, N07AX02, R03DA02, R03DB02, ..."
11,DB00126,Ascorbic acid,"[A11GA01, A11GB01, G01AD03, S01XA15]"
16,DB00136,Calcitriol,"[A11CC04, D05AX03]"
18,DB00140,Riboflavin,"[A11HA04, S01XA26]"
21,DB00146,Calcifediol,"[A11CC06, H05BX05]"
25,DB00158,Folic acid,"[B03AD01, B03AD02, B03AD03, B03AD04, B03AD05, ..."
27,DB00162,Vitamin A,"[A11CA01, D10AD02, R01AX02, S01XA02, V04CB01]"
32,DB00169,Cholecalciferol,"[A11CC05, A11CC55, M05BB03, M05BB04, M05BB05, ..."


## Section 10 — Backfill missing FASTA sequences

`target_uniprot_ids` may now contain accessions gained from the ChEMBL/UniProt backfill passes that
have no matching amino-acid sequence yet (DrugBank only ships sequences for its *own* annotated
targets). For every drug where the sequence count is lower than the UniProt-ID count, we fetch the
missing sequences directly from the UniProt REST FASTA endpoint.

Note `has_fasta` is tracked but **not** folded into `has_all_features` — the core completeness gate in
Section 11 only requires SMILES + ATC + targets. FASTA availability is nice-to-have context, not a hard
requirement, since `biological_overlap.ipynb` re-derives sequences (and everything else target-related)
directly from the DrugBank XML during its own pass anyway.


In [14]:
UNIPROT_FASTA_BASE = "https://rest.uniprot.org/uniprotkb"
FASTA_MAX_WORKERS = 8


def uniprot_fetch_sequence(accession, timeout=10):
    if not accession:
        return None
    url = f"{UNIPROT_FASTA_BASE}/{accession}.fasta"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code == 200 and r.text:
            lines = r.text.strip().splitlines()
            body = lines[1:] if lines and lines[0].startswith(">") else lines
            return "".join(body).strip()
    except Exception:
        pass
    return None


def _resolve_fasta_for_drug(dbid, uniprot_ids, existing_fasta):
    existing_set = set(existing_fasta) if isinstance(existing_fasta, list) else set()
    if not isinstance(uniprot_ids, list) or not uniprot_ids:
        return dbid, sorted(existing_set)
    new_sequences = set(existing_set)
    for acc in uniprot_ids:
        seq = uniprot_fetch_sequence(acc)
        if seq:
            new_sequences.add(seq)
    return dbid, sorted(new_sequences)


def _needs_fasta_backfill(row):
    uids = row["target_uniprot_ids"] if isinstance(row["target_uniprot_ids"], list) else []
    fasta = row["target_fasta_sequences"] if isinstance(row["target_fasta_sequences"], list) else []
    return len(uids) > 0 and len(fasta) < len(uids)  # crude heuristic: fewer sequences than ids


fasta_backfill_rows = per_drug_df.loc[
    per_drug_df.apply(_needs_fasta_backfill, axis=1),
    ["drugbank_id", "target_uniprot_ids", "target_fasta_sequences"],
]

fasta_backfill_results = {}
print(f"Resolving FASTA sequences for {len(fasta_backfill_rows)} drugs using {FASTA_MAX_WORKERS} parallel workers...")

with ThreadPoolExecutor(max_workers=FASTA_MAX_WORKERS) as executor:
    futures = {
        executor.submit(_resolve_fasta_for_drug, row.drugbank_id, row.target_uniprot_ids, row.target_fasta_sequences): row.drugbank_id
        for row in fasta_backfill_rows.itertuples()
    }
    for future in as_completed(futures):
        dbid, seqs = future.result()
        fasta_backfill_results[dbid] = seqs

resolved_fasta_count = sum(1 for seqs in fasta_backfill_results.values() if len(seqs) > 0)
print(f"FASTA backfill resolved sequences for {resolved_fasta_count}/{len(fasta_backfill_results)} drugs")

per_drug_df = per_drug_df.set_index("drugbank_id", drop=False)
for dbid, seqs in fasta_backfill_results.items():
    if seqs:
        per_drug_df.at[dbid, "target_fasta_sequences"] = seqs
per_drug_df = per_drug_df.reset_index(drop=True)

per_drug_df["has_fasta"] = per_drug_df["target_fasta_sequences"].apply(lambda x: not list_is_empty(x))

print("\nAfter FASTA sequence backfill:")
print(per_drug_df[["has_smiles", "has_atc", "has_targets", "has_fasta", "has_all_features"]].mean())


Resolving FASTA sequences for 220 drugs using 8 parallel workers...


FASTA backfill resolved sequences for 214/220 drugs

After FASTA sequence backfill:
has_smiles          0.972557
has_atc             0.687877
has_targets         0.842883
has_fasta           0.791013
has_all_features    0.572979
dtype: float64


## Section 11 — Final completeness gate & save curated datasets

Now that every reasonable backfill source has been exhausted, drugs still missing SMILES, ATC codes, or
targets are discarded for good (`complete_drug_ids`). Both pair datasets are filtered down to only pairs
where **both** drugs are in that complete set, then a **symmetric drug-set gate** additionally requires
every surviving drug to appear in *both* the negative and adverse pair sets -- otherwise a drug that
only ever survives into one population (e.g. all of its documented adverse partners failed the
completeness gate, even though it was also randomly negative-sampled against several complete drugs)
would let a downstream classifier key off drug identity instead of the actual similarity signal. Features
are re-attached one final time, and the result is saved as Parquet — the hand-off point to
`biological_overlap.ipynb`.

We report approved-vs-unapproved drug counts at each step so it stays visible how much of the final
dataset is investigational/unapproved vs. clinically approved.


In [ ]:
complete_drug_ids = set(per_drug_df.loc[per_drug_df["has_all_features"], "drugbank_id"])
approved_lookup = per_drug_df.set_index("drugbank_id")["is_approved"].to_dict()

print(f"Drugs with all core features complete: {len(complete_drug_ids):,} / {len(per_drug_df):,}")
complete_approved = sum(1 for d in complete_drug_ids if approved_lookup.get(d, False))
complete_unapproved = len(complete_drug_ids) - complete_approved
print(f"  -> Approved   : {complete_approved:,}")
print(f"  -> Unapproved : {complete_unapproved:,}")

negative_final_df = filter_pairs_to_ids(negative_df, neg_d1, neg_d2, complete_drug_ids)
adverse_final_df = filter_pairs_to_ids(adverse_df, adv_d1, adv_d2, complete_drug_ids)

print(f"\nNon-interacting: {len(negative_df):,} -> {len(negative_final_df):,}")
print(f"Adverse        : {len(adverse_df):,} -> {len(adverse_final_df):,}")


def unique_drug_ids(df, d1_col, d2_col):
    return set(pd.concat([df[d1_col], df[d2_col]]).astype(str).str.strip().str.upper().unique())


# Symmetric drug-set gate: a drug must appear in BOTH final pair sets, not just pass the
# per-drug completeness gate above -- otherwise a drug that only ever survives into one
# population (e.g. every one of its adverse partners failed the completeness gate, but it was
# also randomly negative-sampled against several complete drugs) lets a downstream classifier
# key off drug identity ("drug X -> always label Y") instead of the actual similarity signal.
# Iterate to a fixed point since dropping a pair can strand its *other* drug if that pair was
# its only surviving partner in this dataset.
for _ in range(10):
    neg_ids = unique_drug_ids(negative_final_df, neg_d1, neg_d2)
    adv_ids = unique_drug_ids(adverse_final_df, adv_d1, adv_d2)
    symmetric_ids = neg_ids & adv_ids
    if symmetric_ids == neg_ids and symmetric_ids == adv_ids:
        break
    negative_final_df = filter_pairs_to_ids(negative_final_df, neg_d1, neg_d2, symmetric_ids)
    adverse_final_df = filter_pairs_to_ids(adverse_final_df, adv_d1, adv_d2, symmetric_ids)
else:
    print("WARNING: symmetric drug-set gate did not converge in 10 passes")

print(f"\nAfter symmetric drug-set gate -- Non-interacting: {len(negative_final_df):,} pairs, Adverse: {len(adverse_final_df):,} pairs")

# Re-attach final features cleanly (SMILES / ATC / targets / FASTA / other_ids only —
# pathway/GO/Pfam enrichment is deliberately NOT computed here; see closing note below).
negative_final_df = attach_features(negative_final_df, neg_d1, neg_d2, per_drug_df.set_index("drugbank_id"), feature_cols)
adverse_final_df = attach_features(adverse_final_df, adv_d1, adv_d2, per_drug_df.set_index("drugbank_id"), feature_cols)


def approved_unapproved_split(ids, lookup):
    approved = sum(1 for d in ids if lookup.get(d, False))
    return approved, len(ids) - approved


neg_final_ids = unique_drug_ids(negative_final_df, neg_d1, neg_d2)
adv_final_ids = unique_drug_ids(adverse_final_df, adv_d1, adv_d2)
union_final_ids = neg_final_ids | adv_final_ids

neg_app, neg_unapp = approved_unapproved_split(neg_final_ids, approved_lookup)
adv_app, adv_unapp = approved_unapproved_split(adv_final_ids, approved_lookup)
union_app, union_unapp = approved_unapproved_split(union_final_ids, approved_lookup)

print(f"\nUnique drugs in final NON-INTERACTING dataset : {len(neg_final_ids):,}  (Approved: {neg_app:,}, Unapproved: {neg_unapp:,})")
print(f"Unique drugs in final ADVERSE         dataset : {len(adv_final_ids):,}  (Approved: {adv_app:,}, Unapproved: {adv_unapp:,})")
print(f"Unique drugs across BOTH (union)              : {len(union_final_ids):,}  (Approved: {union_app:,}, Unapproved: {union_unapp:,})")
print(f"Drugs only in non-interacting: {len(neg_final_ids - adv_final_ids)} | only in adverse: {len(adv_final_ids - neg_final_ids)} (expect 0, 0)")

# Free large intermediates no longer needed before the memory-heavy Arrow/Parquet
# serialization below -- by this point Section 11 alone is holding >10 DataFrames
# simultaneously (the raw 1.1M-row export, drugbank_df, per_drug_df, the pre-gate
# negative/adverse copies, etc.), which is what caused the ArrowMemoryError on the previous run.
import gc

for _name in [
    "raw_all_pairs_df", "positive_conservative_df", "adverse_positive_removed_df",
    "original_adverse_dataset", "original_negative_dataset", "negative_samples_df",
    "documented_pairs", "negative_pairs_seen", "negative_pairs", "all_drugs",
    "drugbank_df", "feat_lookup", "negative_df", "adverse_df",
    "negative_feat_df", "adverse_feat_df", "per_drug_df",
]:
    globals().pop(_name, None)
gc.collect()


NameError: name 'per_drug_df' is not defined

In [20]:
def save_parquet_chunked(df, path, chunk_size=50_000, compression="snappy"):
    """Write `df` to parquet one row-chunk at a time, so the pandas->Arrow conversion never
    holds more than one chunk's buffers in memory at once. A single `to_parquet()` call on
    the full ~470K/~160K-row, FASTA-sequence-heavy DataFrame was hitting ArrowMemoryError even
    after freeing every other large intermediate (previous cell) -- the culprit is the size of
    the *single* Arrow conversion, not lingering unrelated objects, so chunking is the fix. Kept
    in its own cell so a failed/retried save doesn't require re-running the (already-freed,
    expensive-to-rebuild) upstream computation above.
    """
    import pyarrow as pa
    import pyarrow.parquet as pq

    writer = None
    try:
        for start in range(0, len(df), chunk_size):
            chunk = df.iloc[start:start + chunk_size]
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(path, table.schema, compression=compression)
            writer.write_table(table)
            del chunk, table
    finally:
        if writer is not None:
            writer.close()


os.makedirs(CONFIG["output_dir"], exist_ok=True)
save_parquet_chunked(adverse_final_df, os.path.join(CONFIG["output_dir"], "adverse_final_df.parquet"))
gc.collect()  # release the just-written Arrow buffers before the second (comparably large) write
save_parquet_chunked(negative_final_df, os.path.join(CONFIG["output_dir"], "negative_final_df.parquet"))
print(f"\nSaved adverse_final_df.parquet and negative_final_df.parquet to:\n  {CONFIG['output_dir']}")

adverse_final_df.head()



Saved adverse_final_df.parquet and negative_final_df.parquet to:
  C:\Users\ashto\ddi-prediction\data\sample


,drug1_id,drug1_name,drug2_id,drug2_name,description,pair_key,is_therapeutic_efficacy,smiles_1,smiles_2,atc_codes_list_1,atc_codes_list_2,target_uniprot_ids_1,target_uniprot_ids_2,target_fasta_sequences_1,target_fasta_sequences_2,target_other_ids_1,target_other_ids_2
0,DB00006,Bivalirudin,DB06605,Apixaban,Apixaban may increase the anticoagulant activi...,"('DB00006', 'DB06605')",False,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,COC1=CC=C(C=C1)N1N=C(C(N)=O)C2=C1C(=O)N(CC2)C1...,[B01AE06],[B01AF02],[P00734],[P00742],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MGRPLHLVLLSASLAGLLLLGESLFIRREQANNILARVTRANSFL...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[182841, 2359, Coagulation factor X, F10, FA10..."
1,DB00006,Bivalirudin,DB06695,Dabigatran etexilate,Dabigatran etexilate may increase the anticoag...,"('DB00006', 'DB06695')",False,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,CCCCCCOC(=O)\N=C(\N)C1=CC=C(NCC2=NC3=C(C=CC(=C...,[B01AE06],[B01AE07],[P00734],[P00734],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[2362, 339641, F2, HGNC:3535, M17262, Prothrom..."
2,DB00006,Bivalirudin,DB01254,Dasatinib,The risk or severity of bleeding and hemorrhag...,"('DB00006', 'DB01254')",False,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,CC1=NC(NC2=NC=C(S2)C(=O)NC2=C(C)C=CC=C2Cl)=CC(...,[B01AE06],[L01EA02],[P00734],"[P00519, P06239, P06241, P07947, P07948, P0961...",[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MAAVILESIFLKRSQQKKKTSPLNFKKRLFLLTVHKLSYYEYDFE...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[10635153, 11526573, 1499, 178993, 1804, 1805,..."
3,DB00006,Bivalirudin,DB01609,Deferasirox,The risk or severity of gastrointestinal bleed...,"('DB00006', 'DB01609')",False,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,OC(=O)C1=CC=C(C=C1)N1N=C(N=C1C1=CC=CC=C1O)C1=C...,[B01AE06],[V03AC03],[P00734],[],[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[],"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...",[Iron]
4,DB00006,Bivalirudin,DB01586,Ursodeoxycholic acid,The risk or severity of bleeding and bruising ...,"('DB00006', 'DB01586')",False,CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@...,[H][C@@]1(CC[C@@]2([H])[C@]3([H])[C@@H](O)C[C@...,[B01AE06],[A05AA02],[P00734],"[P52895, P53004, Q96RI1, Q9UGH3]",[MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRAN...,[MDSKYQCVKLNDGHFMPVLGFGTYAPAEVPKSKALEAVKLAIEAG...,"[2362, 339641, F2, HGNC:3535, M17262, Prothrom...","[1246749, 1546084, 531160, 603, AK1C2_HUMAN, A..."


## Where to go from here

`data/sample/adverse_final_df.parquet` and `data/sample/negative_final_df.parquet` are the outputs of
this notebook: drug-drug pairs where both drugs have a verified SMILES string, at least one ATC code,
and at least one target identifier. **These two files are the single source of truth for every
similarity notebook downstream** -- each one loads them directly from `data/sample/` (no other copies
or OneDrive paths should be used):

- [`notebooks/h1_biological_overlap/biological_overlap.ipynb`](../h1_biological_overlap/biological_overlap.ipynb) --
  rebuilds a much richer biological profile directly from the DrugBank XML + ChEMBL for every drug
  referenced in the two files above:
  - `L0`/`L1`/`L2` protein-role ladder (targets → + enzymes/transporters → + carriers), with `T(d)`
    pooled from **both** DrugBank and ChEMBL mechanism/bioactivity data.
  - Gene Ontology terms split by aspect (`GO_MF`, `GO_BP`, `GO_CC`).
  - Native SMPDB pathway membership (`Φ_native`) and an inferred two-hop pathway-neighbor expansion
    (`Φ_infer`).
  - Pfam domain sets.
  - Jaccard similarity + directional Tversky containment scores for every layer above, for both the
    adverse and negative pair sets.
- [`notebooks/h2_pharmacological_similarity/atc_similiarity_notebook.ipynb`](../h2_pharmacological_similarity/atc_similiarity_notebook.ipynb) --
  computes ATC hierarchy (Mean-of-Max) similarity from each pair's `atc_codes_list_1`/`atc_codes_list_2`.
- [`notebooks/h3_structural_similarity/structural_sim_notebook.ipynb`](../h3_structural_similarity/structural_sim_notebook.ipynb) --
  computes Morgan-fingerprint Tanimoto similarity from each pair's `smiles_1`/`smiles_2`.

That separation is intentional: this notebook owns *"which pairs exist and do both drugs have a usable
identity"*, and the three notebooks above each own one independent *"how similar are they by this one
lens"* question. Keeping them separate means re-running the (slow, network-bound) backfill logic here
doesn't require re-deriving any of the three similarity scores, and vice versa -- and all three can be
regenerated in any order (or in parallel) once this notebook's two Parquet files exist.

Their outputs, in turn, feed [`notebooks/merging_into_final_drug_pair_feature_table.ipynb`](../merging_into_final_drug_pair_feature_table.ipynb)
(joins all three similarity scores into one table) and each `hN_*/network_generation.ipynb` (builds the
`networkx` graphs used for node2vec).
